# 第16课：Pydantic 结构化 Prompt

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter16_Pydantic结构化Prompt_课后练习.ipynb](chapter16_Pydantic结构化Prompt_课后练习.ipynb)。

**阶段定位**：阶段二 · 任务一 1.2 / 15分。15分

模型愿意“配合输出 JSON”不等于字段合法。本课把 JSON Schema 写进 Prompt，再用 Pydantic 当考官。任务一 1.2 按三档给分。

## 学习目标

1. 用 BaseModel 与 Field 定义输出模式，并能导出 JSON Schema。
2. 基础：扁平用户信息，手机号去空格且正则通过。
3. 进阶：嵌套工单，状态枚举，total 与明细一致。
4. 高难：自定义 field_validator 归一化脏金额，无法解析则拦截。

## 学习知识点

| 基础 5分 | 进阶 5分 | 高难 5分 |
| --- | --- | --- |
| UserProfile | TicketAnalysis | DirtyAmount |
| 扁平字段 | 多行商品嵌套 | @field_validator |
| 手机号 11 位 | status 枚举 + total | ￥1,299.00 → 1299.0 |

## 基础回顾与案例提问

1. **R.1** Prompt 里写了“必须 11 位手机号”，模型仍输出带空格的号码。谁负责去空格？
2. **R.2** items 两行金额合计 3297，模型却写 total=3000。应过还是失败？
3. **R.3** 金额字段是“免费送”，校验器应归一成 0 还是 raise？

本课使用 pydantic v2 的 `field_validator` / `model_validator`。不要使用 v1 的 `@validator`。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. Schema 注入 Prompt

### 理论知识

**先有结构，再让模型填空。** `model_json_schema()` 可放进 system 消息。模型输出后仍必须 `model_validate`。

### 案例：导出一段 Schema


In [ ]:
from agent_lab.schemas import UserProfile
print(list(UserProfile.model_json_schema()["properties"]))


### 讲解

Field(description=...) 会出现在 schema 里，等于把注释写给模型看。不要只在中文题面里解释字段。

### 易错点与练习

1. **K1.1** 为什么“只在 Prompt 里骂模型要守规矩”不够？
2. **K1.2** properties 的键应是字段名还是中文标签？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 基础：扁平用户

### 理论知识

**单层字段 + 正则。** name / phone / age。phone 允许带空格，校验器去掉后必须是 1 开头 11 位。

### 案例：解析一条用户


In [ ]:
from agent_lab.schemas import UserProfile, parse_model
ok, profile = parse_model(UserProfile, {"name": "张三", "phone": "138 0013 8000", "age": 21})
print(ok, profile)


### 讲解

parse_model 捕获 ValidationError，返回 (False, exc) 而不是炸内核。这与 Tool 的异常隔离同一哲学。

### 易错点与练习

1. **K2.1** age=-1 会失败在哪个约束？
2. **K2.2** phone 写成 1380013800（10 位）应否通过？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 进阶：嵌套工单

### 理论知识

**LineItem 列表 + status 枚举 + total 交叉检验。** 这是工单明细分析，不是再做一个扁平 dict。

### 案例：合法工单


In [ ]:
from agent_lab.schemas import TicketAnalysis, parse_model
payload = {
    "ticket_id": "T-10086",
    "status": "open",
    "items": [
        {"sku": "WIDGET-X", "qty": 2, "price": 1299.0},
        {"sku": "WIDGET-MINI", "qty": 1, "price": 699.0},
    ],
    "total": 3297.0,
}
ok, ticket = parse_model(TicketAnalysis, payload)
print(ok, ticket.total if ok else ticket)


### 讲解

2*1299 + 699 = 3297。model_validator 在全部字段就位后比较四舍五入到分。

### 易错点与练习

1. **K3.1** status='OPEN' 能过枚举吗？
2. **K3.2** qty=0 为什么应失败？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 合计不一致

### 理论知识

**进阶档的区分度在交叉校验。** 模型算错 total 时必须失败，不能“差不多就行”。

### 案例：故意写错 total


In [ ]:
bad = dict(payload)
bad["total"] = 1.0
ok, err = parse_model(TicketAnalysis, bad)
print(ok)
print(str(err.errors()[0]["msg"])[:60] if not ok else err)


### 讲解

客服场景里金额不一致比缺一个逗号更危险。Schema 要拦业务不变量，不只拦类型。

### 易错点与练习

1. **K4.1** 若只校验类型不校验合计，漏掉的是哪类事故？
2. **K4.2** 为什么用 round(..., 2) 再比较？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 高难：归一化脏金额

### 理论知识

**before 校验器先把字符串变成数。** 去掉 ￥、¥、逗号、元、空格；仍不是数字则 ValueError。

### 案例：￥1,299.00


In [ ]:
from agent_lab.schemas import DirtyAmount, parse_model
ok, dirty = parse_model(DirtyAmount, {"amount": "￥1,299.00", "currency": "CNY"})
print(ok, dirty)


### 讲解

这是容错，不是放水：能规则化的规则化，不能的必须拦截。

### 易错点与练习

1. **K5.1** mode='after' 时还能处理带 ￥ 的字符串吗？先预测。
2. **K5.2** currency 缺省 CNY，传入 'usd' 小写是否本课要标准化？本课不强制。

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 对抗性脏数据

### 理论知识

**“免费送”不得变成 0 元。** 静默成 0 会制造假促销。

### 案例：拦截


In [ ]:
ok, bad = parse_model(DirtyAmount, {"amount": "免费送", "currency": "CNY"})
print(ok)
if not ok:
    print(bad.errors()[0]["msg"])


### 讲解

高难 5 分同时看两面：能纠偏的纠偏，该拒绝的拒绝。

### 易错点与练习

1. **K6.1** 若有人在校验器里 `return 0` 处理一切异常，违反了哪条评分精神？
2. **K6.2** 空字符串应拦截还是当 0？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 与模型输出对接

### 理论知识

**Fake 客户端见到“用户信息”“工单”“脏数据”会返回对应 JSON。** 真实模型则依赖你把 schema 塞进 Prompt。

### 案例：从聊天拿 JSON 再校验


In [ ]:
import json
from agent_lab.mock_openai import FakeOpenAI
from agent_lab.schemas import UserProfile
raw = FakeOpenAI().chat.completions.create(
    model="qwen2.5",
    messages=[{"role": "user", "content": "请抽取用户信息"}],
)["choices"][0]["message"]["content"]
print(UserProfile.model_validate(json.loads(raw)))


### 讲解

课堂 Fake 保证可重复。换真实模型时，失败样例会变多，这正是 Schema 的价值。

### 易错点与练习

1. **K7.1** 为什么不让模型直接返回 Python 对象？
2. **K7.2** json.loads 失败和 ValidationError 应分成两种日志吗？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 15 分评分对照

### 理论知识

**三例各 5 分，不能互相替代。** 只会扁平用户最高 5 分。

### 案例：三连


In [ ]:
print("基础", parse_model(UserProfile, {"name": "张三", "phone": "13800138000", "age": 21})[0])
print("进阶见 TicketAnalysis")
print("高难见 DirtyAmount 与免费送")


### 讲解

experiment.py 四次断言：扁平成功、工单成功、脏金额成功、免费送失败。

### 易错点与练习

1. **K8.1** 漏做对抗拦截还能否拿满高难 5 分？
2. **K8.2** 把校验写在 if 字符串里而不用 Pydantic，是否符合题面？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：三级梯度

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　扁平用户

解析带空格手机号，打印规范化后的 phone。


In [ ]:
# P1.1: UserProfile.


### P1.2　嵌套工单

提交合法工单并打印 total；可选再交一份 total 错误的失败输出。


In [ ]:
# P1.2: TicketAnalysis.


### P1.3　脏金额

￥1,299.00 成功；免费送失败。


In [ ]:
# P1.3: DirtyAmount success and reject.


课后请打开 [chapter16_Pydantic结构化Prompt_课后练习.ipynb](chapter16_Pydantic结构化Prompt_课后练习.ipynb)。P1 基础、P2 进阶、P3 选做自己写一个校验器。
